In [27]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd

In [20]:
load_dotenv()
api_key = os.getenv("USDA_API_KEY")

In [21]:
params = {
    "dataType": ["Survey (FNDDS)"],
    "pageSize": 5,
    "pageNumber": 1,
}

resp = requests.post(
    "https://api.nal.usda.gov/fdc/v1/foods/list",
    params={"api_key": api_key},
    json=params
)
print(resp.status_code)
print(resp.text)

200
[{"fdcId":2706337,"description":"Abalone","dataType":"Survey (FNDDS)","publicationDate":"2024-10-31","foodCode":"26301110","foodNutrients":[{"number":"203","name":"Protein","amount":21.2,"unitName":"G"},{"number":"204","name":"Total lipid (fat)","amount":6.12,"unitName":"G"},{"number":"205","name":"Carbohydrate, by difference","amount":7.46,"unitName":"G"},{"number":"208","name":"Energy","amount":177,"unitName":"KCAL"},{"number":"221","name":"Alcohol, ethyl","amount":0E-8,"unitName":"G"},{"number":"255","name":"Water","amount":62.7,"unitName":"G"},{"number":"262","name":"Caffeine","amount":0E-8,"unitName":"MG"},{"number":"263","name":"Theobromine","amount":0E-8,"unitName":"MG"},{"number":"269","name":"Total Sugars","amount":0E-8,"unitName":"G"},{"number":"291","name":"Fiber, total dietary","amount":0E-8,"unitName":"G"},{"number":"301","name":"Calcium, Ca","amount":39.0,"unitName":"MG"},{"number":"303","name":"Iron, Fe","amount":3.97,"unitName":"MG"},{"number":"304","name":"Magnesiu

Check to see if stored in GCS:

In [23]:
from google.oauth2 import service_account
from google.cloud import storage
import json
import os
from dotenv import load_dotenv

load_dotenv()
sa_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
proj_id = os.getenv("GCP_PROJECT_ID")

credentials = service_account.Credentials.from_service_account_file(sa_key)
client = storage.Client(project=proj_id, credentials=credentials)

# list all buckets
print("YOUR BUCKETS:")
for bucket in client.list_buckets():
    print(bucket.name)

YOUR BUCKETS:
foundation-foods-food-health-pipeline
legacy-foods-food-health-pipeline
survey-foods-food-health-pipeline


In [24]:
# look inside the survey bucket
bucket = client.get_bucket("survey-foods-food-health-pipeline")

print("FILES IN SURVEY BUCKET")
for blob in bucket.list_blobs():
    print(blob.name, "—", blob.size, "bytes")

FILES IN SURVEY BUCKET
survey_all/2026-03-16.json — 28362516 bytes


In [ ]:
# read first record from the survey file
blob = list(bucket.list_blobs())[0]
data = json.loads(blob.download_as_string())

print(f"Total records: {len(data)}")
print(f"\nFirst record:")
data[0]

Total records: 5432

First record:


{'fdcId': 2706337,
 'description': 'Abalone',
 'dataType': 'Survey (FNDDS)',
 'publicationDate': '2024-10-31',
 'foodCode': '26301110',
 'foodNutrients': [{'number': '203',
   'name': 'Protein',
   'amount': 21.2,
   'unitName': 'G'},
  {'number': '204',
   'name': 'Total lipid (fat)',
   'amount': 6.12,
   'unitName': 'G'},
  {'number': '205',
   'name': 'Carbohydrate, by difference',
   'amount': 7.46,
   'unitName': 'G'},
  {'number': '208', 'name': 'Energy', 'amount': 177, 'unitName': 'KCAL'},
  {'number': '221', 'name': 'Alcohol, ethyl', 'amount': 0.0, 'unitName': 'G'},
  {'number': '255', 'name': 'Water', 'amount': 62.7, 'unitName': 'G'},
  {'number': '262', 'name': 'Caffeine', 'amount': 0.0, 'unitName': 'MG'},
  {'number': '263', 'name': 'Theobromine', 'amount': 0.0, 'unitName': 'MG'},
  {'number': '269', 'name': 'Total Sugars', 'amount': 0.0, 'unitName': 'G'},
  {'number': '291',
   'name': 'Fiber, total dietary',
   'amount': 0.0,
   'unitName': 'G'},
  {'number': '301', 'name

In [28]:
# survey data
survey_bucket = client.get_bucket("survey-foods-food-health-pipeline")
survey_blob = list(survey_bucket.list_blobs())[0]
survey_data = json.loads(survey_blob.download_as_string())
survey_df = pd.DataFrame(survey_data)
print(f"Survey records: {len(survey_df)}")
survey_df.head()

Survey records: 5432


,fdcId,description,dataType,publicationDate,foodCode,foodNutrients
0,2706337,Abalone,Survey (FNDDS),2024-10-31,26301110,"[{'number': '203', 'name': 'Protein', 'amount'..."
1,2708809,"Adobo, with noodles",Survey (FNDDS),2024-10-31,58137300,"[{'number': '203', 'name': 'Protein', 'amount'..."
2,2708958,"Adobo, with rice",Survey (FNDDS),2024-10-31,58150530,"[{'number': '203', 'name': 'Protein', 'amount'..."
3,2710282,Agave liquid sweetener,Survey (FNDDS),2024-10-31,91302020,"[{'number': '203', 'name': 'Protein', 'amount'..."
4,2710681,Alcoholic coffee drink,Survey (FNDDS),2024-10-31,93301400,"[{'number': '203', 'name': 'Protein', 'amount'..."


In [29]:
# legacy data
legacy_bucket = client.get_bucket("legacy-foods-food-health-pipeline")
legacy_blob = list(legacy_bucket.list_blobs())[0]
legacy_data = json.loads(legacy_blob.download_as_string())
legacy_df = pd.DataFrame(legacy_data)
print(f"Legacy records: {len(legacy_df)}")
legacy_df.head()

Legacy records: 7793


,fdcId,description,dataType,publicationDate,ndbNumber,foodNutrients
0,167782,"Abiyuch, raw",SR Legacy,2019-04-01,9427,"[{'number': '318', 'name': 'Vitamin A, IU', 'a..."
1,171687,"Acerola juice, raw",SR Legacy,2019-04-01,9002,"[{'number': '268', 'name': 'Energy', 'amount':..."
2,171686,"Acerola, (west indian cherry), raw",SR Legacy,2019-04-01,9001,"[{'number': '268', 'name': 'Energy', 'amount':..."
3,168061,Acorn stew (Apache),SR Legacy,2019-04-01,35182,"[{'number': '429', 'name': 'Vitamin K (Dihydro..."
4,168992,"Agave, cooked (Southwest)",SR Legacy,2019-04-01,35193,"[{'number': '324', 'name': 'Vitamin D (D2 + D3..."


In [30]:
# foundation data
foundation_bucket = client.get_bucket("foundation-foods-food-health-pipeline")
foundation_blob = list(foundation_bucket.list_blobs())[0]
foundation_data = json.loads(foundation_blob.download_as_string())
foundation_df = pd.DataFrame(foundation_data)
print(f"Foundation records: {len(foundation_df)}")
foundation_df.head()

Foundation records: 365


,fdcId,description,dataType,publicationDate,ndbNumber,foodNutrients
0,2262074,"Almond butter, creamy",Foundation,2022-04-28,12195,"[{'number': '717', 'name': 'Daidzin', 'amount'..."
1,2257045,"Almond milk, unsweetened, plain, refrigerated",Foundation,2022-04-28,100276,"[{'number': '404', 'name': 'Thiamin', 'amount'..."
2,1999631,"Almond milk, unsweetened, plain, shelf stable",Foundation,2021-10-28,14091,"[{'number': '631', 'name': 'PUFA 22:5 n-3 (DPA..."
3,2747652,"Anchovies, canned in olive oil, with salt, dra...",Foundation,2025-12-18,100412,"[{'number': '303', 'name': 'Iron, Fe', 'amount..."
4,2003590,"Apple juice, with added vitamin C, from concen...",Foundation,2021-10-28,9400,"[{'number': '303', 'name': 'Iron, Fe', 'amount..."


In [33]:
# look at nutrients for abalone in a clean dataframe
first_food = survey_df.iloc[0]
print(f"Food: {first_food['description']}")
print(f"Calories: {[n['amount'] for n in first_food['foodNutrients'] if n['name'] == 'Energy']}")
print()

# convert nutrients to a small dataframe
nutrients_df = pd.DataFrame(first_food['foodNutrients'])
nutrients_df[['name', 'amount', 'unitName']]

Food: Abalone
Calories: [177]



,name,amount,unitName
0,Protein,21.200,G
1,Total lipid (fat),6.120,G
2,"Carbohydrate, by difference",7.460,G
3,Energy,177.000,KCAL
4,"Alcohol, ethyl",0.000,G
...,...,...,...
60,PUFA 20:5 n-3 (EPA),0.061,G
61,MUFA 22:1,0.000,G
62,PUFA 22:5 n-3 (DPA),0.051,G
63,"Fatty acids, total monounsaturated",2.330,G
